In [1]:
# =========================
# Finance + Text combined model
# =========================

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


# =========================
# 1. Load data
# =========================
X_TEXT_PATH = "X_text.csv"
X_FIN_PATH = "X_fin_pca.csv"
Y_PATH = "deepseek_Y_quantile_final.csv"
BRIDGE_PATH = "Final_Matched_Universe_Full.csv"

x_text = pd.read_csv(X_TEXT_PATH)
x_fin = pd.read_csv(X_FIN_PATH)
y = pd.read_csv(Y_PATH)
bridge = pd.read_csv(BRIDGE_PATH)

print("x_text shape:", x_text.shape)
print("x_fin shape:", x_fin.shape)
print("y shape:", y.shape)
print("bridge shape:", bridge.shape)


# =========================
# 2. Detect key columns
# =========================
def find_col(df, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

xtext_companyid_col = find_col(x_text, ["companyid", "company_id"])
xtext_companyname_col = find_col(x_text, ["companyname", "company_name"])

xfin_ticker_col = find_col(x_fin, ["tic", "ticker"])

y_ticker_col = find_col(y, ["ticker", "tic"])
y_label_col = find_col(y, ["label"])

bridge_companyid_col = find_col(bridge, ["companyid", "company_id"])
bridge_ticker_col = find_col(bridge, ["ticker", "tic"])
bridge_companyname_col = find_col(bridge, ["companyname", "company_name", "title"])

print("\nDetected columns:")
print("x_text companyid:", xtext_companyid_col)
print("x_fin ticker:", xfin_ticker_col)
print("y ticker:", y_ticker_col)
print("y label:", y_label_col)
print("bridge companyid:", bridge_companyid_col)
print("bridge ticker:", bridge_ticker_col)

if any(v is None for v in [xtext_companyid_col, xfin_ticker_col, y_ticker_col, y_label_col, bridge_companyid_col, bridge_ticker_col]):
    raise ValueError("Some key columns were not detected. Please check the printed column names and edit them manually.")


# =========================
# 3. Prepare Y
# =========================
y_keep = y[[y_ticker_col, y_label_col]].drop_duplicates().copy()

label_map = {
    "Vulnerable": 1,
    "Resilient": 0
}
y_keep["label_binary"] = y_keep[y_label_col].map(label_map)
y_keep = y_keep.rename(columns={y_ticker_col: "ticker"})

print("\nY label counts:")
print(y_keep[y_label_col].value_counts(dropna=False))
print(y_keep["label_binary"].value_counts(dropna=False))


# =========================
# 4. Prepare bridge
# =========================
bridge_keep_cols = [bridge_companyid_col, bridge_ticker_col]
if bridge_companyname_col is not None:
    bridge_keep_cols.append(bridge_companyname_col)

bridge_keep = bridge[bridge_keep_cols].drop_duplicates().copy()
bridge_keep = bridge_keep.rename(columns={
    bridge_companyid_col: "companyid_bridge",
    bridge_ticker_col: "ticker"
})

print("\nBridge preview:")
print(bridge_keep.head())


# =========================
# 5. Merge X_text with bridge
# =========================
text_merged = x_text.merge(
    bridge_keep,
    left_on=xtext_companyid_col,
    right_on="companyid_bridge",
    how="inner"
)

print("\nAfter X_text + bridge:", text_merged.shape)


# =========================
# 6. Prepare finance table
# =========================
x_fin_renamed = x_fin.rename(columns={xfin_ticker_col: "ticker"})


# =========================
# 7. Merge text + finance + Y
# =========================
model_all = text_merged.merge(
    x_fin_renamed,
    on="ticker",
    how="inner",
    suffixes=("_text", "_fin")
).merge(
    y_keep,
    on="ticker",
    how="inner"
)

print("\nMerged combined dataset shape:", model_all.shape)
print("\nMerged label counts:")
print(model_all[y_label_col].value_counts(dropna=False))

print("\nDuplicate tickers in merged data:", model_all["ticker"].duplicated().sum())
print("\nMissing values (top 30):")
print(model_all.isna().sum().sort_values(ascending=False).head(30))


# =========================
# 8. Define feature columns
# =========================
finance_feature_cols = [c for c in model_all.columns if c.startswith("PC")]

drop_text_cols = {
    # IDs
    xtext_companyid_col,
    xtext_companyname_col,
    "companyid_bridge",
    "ticker",
    "companyname",
    "company_name",
    "title",
    "gvkey",
    "datadate",

    # labels
    y_label_col,
    "label_binary",

    # highly redundant text features
    "pct_positive",
    "pct_negative",
    "pct_neutral",
    "mean_p_positive",
    "mean_p_negative",
    "mean_p_neutral",
    "ai_keyword_count",
}

text_feature_cols = [
    c for c in model_all.columns
    if (c not in drop_text_cols) and (not c.startswith("PC"))
]

combined_feature_cols = finance_feature_cols + text_feature_cols

print("\nFinance features:")
print(finance_feature_cols)

print("\nText features:")
print(text_feature_cols)

print("\nTotal number of combined features:", len(combined_feature_cols))

X = model_all[combined_feature_cols].copy()
y_binary = model_all["label_binary"].copy()

print("\nX shape:", X.shape)
print("y shape:", y_binary.shape)


# =========================
# 9. Define evaluation function
# =========================
def evaluate_model(X, y, model_type="logit"):
    """
    Evaluate a model using 5-fold stratified cross-validation.
    Returns accuracy, F1, and ROC-AUC.
    """

    if model_type == "logit":
        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=42
            ))
        ])
    elif model_type == "rf":
        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=3,
                class_weight="balanced",
                random_state=42
            ))
        ])
    else:
        raise ValueError("model_type must be either 'logit' or 'rf'")

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scoring = {
        "accuracy": "accuracy",
        "f1": "f1",
        "roc_auc": "roc_auc"
    }

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    result = {
        "model_type": model_type,
        "n_features": X.shape[1],
        "accuracy_mean": np.mean(scores["test_accuracy"]),
        "accuracy_std": np.std(scores["test_accuracy"]),
        "f1_mean": np.mean(scores["test_f1"]),
        "f1_std": np.std(scores["test_f1"]),
        "roc_auc_mean": np.mean(scores["test_roc_auc"]),
        "roc_auc_std": np.std(scores["test_roc_auc"]),
    }
    return result


# =========================
# 10. Run two baseline models
# =========================
results = []

results.append(evaluate_model(X, y_binary, model_type="logit"))
results.append(evaluate_model(X, y_binary, model_type="rf"))

results_df = pd.DataFrame(results)

print("\n=== Finance + Text baseline results ===")
print(results_df.round(4).to_string(index=False))


# =========================
# 11. Save outputs
# =========================
model_all.to_csv("model_finance_text_combined.csv", index=False)
results_df.to_csv("baseline_results_finance_text_combined.csv", index=False)

print("\nSaved files:")
print("  model_finance_text_combined.csv")
print("  baseline_results_finance_text_combined.csv")

x_text shape: (400, 28)
x_fin shape: (403, 11)
y shape: (158, 17)
bridge shape: (404, 6)

Detected columns:
x_text companyid: companyid
x_fin ticker: tic
y ticker: ticker
y label: label
bridge companyid: companyid
bridge ticker: ticker

Y label counts:
label
Vulnerable    79
Resilient     79
Name: count, dtype: int64
label_binary
1    79
0    79
Name: count, dtype: int64

Bridge preview:
   companyid_bridge ticker            company_name
0         608901501     TE          T1 Energy Inc.
1        1859254185    WAY   Waystar Holding Corp.
2        1801104960   NATL         NCR Atleos Corp
3        1816580458   UMAC  Unusual Machines, Inc.
4         289284240   RBRK            Rubrik, Inc.

After X_text + bridge: (400, 31)

Merged combined dataset shape: (154, 43)

Merged label counts:
label
Resilient     78
Vulnerable    76
Name: count, dtype: int64

Duplicate tickers in merged data: 0

Missing values (top 30):
companyid                                       0
datadate                  